In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os, sys
import numpy as np 
import pandas as pd 
import matplotlib.pyplot as plt
import pymc as pm
import arviz as az
import pytensor
import pytensor.tensor as pt
from sklearn.metrics import r2_score
from tqdm import tqdm

sys.path.append(os.path.abspath(os.path.join(os.getcwd(), os.pardir, 'src')))
from dataloadermaker import DataLoaderMaker
import vis_utils as vu
import probabilistic_utils as pu
from collections import defaultdict, Counter

USE_MACOS = True 
if USE_MACOS:
    pytensor.config.cxx = ""

In [3]:
fp_proc_moth_data = '/Users/tplas/data/2025-06-16 moth data Natalie/1994-2019_field-d50.csv'
assert os.path.exists(fp_proc_moth_data)

df_proc = pd.read_csv(fp_proc_moth_data, sep=';')
df_proc

,YearCatch,YearHatch,AreaShortName,Site,Tree,NovemberDate,TubeNumber,D50Calc
0,1994,1995,HV,98,4,15,1,15
1,1994,1995,HV,98,44,15,3,8
2,1994,1995,HV,98,11,15,4,1
3,1994,1995,HV,98,41,15,6,8
4,1994,1995,HV,98,41,15,7,8
...,...,...,...,...,...,...,...,...
5787,2018,2019,DO,1,11,40,599,"7,4"
5788,2018,2019,DO,4,9,40,624,"17,6"
5789,2018,2019,DO,5,1,40,628,"5,9"
5790,2018,2019,DO,5,4,40,632,"3,8"


In [4]:
fp_raw_moth_data = '/Users/tplas/data/2025-10-07 moth raw data 2023/Qry_NumberOfCaterpillarsPerClutchPerDay.txt'
assert os.path.exists(fp_raw_moth_data)

## read as csv 
df_raw = pd.read_csv(fp_raw_moth_data, sep=';')
df_raw

,YearCatch,YearHatch,AreaShortName,Site,Tree,NovemberDate,TubeID,ExperimentName,AprilDay,Caterpillars
0,1994,1995,HV,98,4,15,5944,CG1995,-14,0.0
1,1994,1995,HV,98,4,15,5944,CG1995,-1,0.0
2,1994,1995,HV,98,4,15,5944,CG1995,5,0.0
3,1994,1995,HV,98,4,15,5944,CG1995,11,0.0
4,1994,1995,HV,98,4,15,5944,CG1995,19,30.0
...,...,...,...,...,...,...,...,...,...,...
119003,2024,2025,WA,2,12,25,21400,CG2025,10,6.0
119004,2024,2025,WA,2,12,25,21400,CG2025,16,8.0
119005,2024,2025,WA,2,12,25,21400,CG2025,17,1.0
119006,2024,2025,WA,2,12,25,21400,CG2025,22,0.0


In [13]:
df_raw.TubeID.unique()[440:480]

array([ 978,  979,  981,  982,  983,  984,  985,  986,  987,  988, 6079,
        989,  990,  991,  992,  993,  994,  995,  996,  997,  998,  999,
       1000, 1001, 1002, 1003, 1004, 1005, 1006, 1007, 1008, 1009, 1037,
       1038, 1039, 1040, 1041, 1042, 1043, 1044])

In [18]:
df_raw[df_raw.TubeID == 1045]

,YearCatch,YearHatch,AreaShortName,Site,Tree,NovemberDate,TubeID,ExperimentName,AprilDay,Caterpillars
4628,1997,1998,DO,1,3,33,1045,CG1998,-11,0.0
4629,1997,1998,DO,1,3,33,1045,CG1998,-6,0.0
4630,1997,1998,DO,1,3,33,1045,CG1998,-1,0.0
4631,1997,1998,DO,1,3,33,1045,CG1998,6,6.0
4632,1997,1998,DO,1,3,33,1045,CG1998,9,10.0
4633,1997,1998,DO,1,3,33,1045,CG1998,14,15.0
4634,1997,1998,DO,1,3,33,1045,CG1998,17,0.0
4635,1997,1998,DO,1,3,33,1045,CG1998,20,0.0
4636,1997,1998,DO,1,3,33,1045,CG1998,24,0.0
4637,1997,1998,DO,1,3,33,1045,CG1998,27,0.0


In [17]:
df_raw[np.logical_and(df_raw.YearHatch == 1998, df_raw.Tree == 3)]

,YearCatch,YearHatch,AreaShortName,Site,Tree,NovemberDate,TubeID,ExperimentName,AprilDay,Caterpillars
4582,1997,1998,DO,1,3,13,1040,CG1998,-11,0.0
4583,1997,1998,DO,1,3,13,1040,CG1998,-6,0.0
4584,1997,1998,DO,1,3,13,1040,CG1998,-1,0.0
4585,1997,1998,DO,1,3,13,1040,CG1998,6,0.0
4586,1997,1998,DO,1,3,13,1040,CG1998,9,0.0
...,...,...,...,...,...,...,...,...,...,...
4863,1997,1998,DO,6,3,33,1068,CG1998,14,2.0
4864,1997,1998,DO,6,3,33,1068,CG1998,17,5.0
4865,1997,1998,DO,6,3,33,1068,CG1998,20,0.0
4866,1997,1998,DO,6,3,33,1068,CG1998,24,0.0


In [16]:
df_proc[np.logical_and(df_proc.YearHatch == 1998, df_proc.Tree == 3)]

,YearCatch,YearHatch,AreaShortName,Site,Tree,NovemberDate,TubeNumber,D50Calc
429,1997,1998,DO,1,3,27,21,"3,6"
430,1997,1998,DO,1,3,27,23,"11,5"
432,1997,1998,DO,4,3,27,25,"11,5"
433,1997,1998,DO,4,3,27,26,6
434,1997,1998,DO,4,3,27,28,6
438,1997,1998,DO,6,3,27,32,"8,4"
439,1997,1998,DO,6,3,27,33,"14,6"
462,1997,1998,DO,1,3,33,68,"8,8"
464,1997,1998,DO,4,3,33,71,"14,9"


In [43]:
def calculate_d50(df, verbose=0):
    max_cp = int(df.Caterpillars.max())
    if max_cp == 0.0:
        if verbose:
            print("WARNING: max_cp is 0.0"
                  f" for TubeID {df.TubeID.values[0]}")
        return np.nan
    
    half_cp = max_cp / 2

    ## get the first day where the caterpillars are above half_cp
    df_half_top = df[df.Caterpillars >= half_cp]
    day_top = df_half_top.AprilDay.min()
    count_top = df_half_top[df_half_top.AprilDay == day_top].Caterpillars.values[0]
    # count_top = df_half_top.Caterpillars.min()
    # day_top = df_half_top[df_half_top.Caterpillars == count_top].AprilDay.values[0]

    ## get the last day where the caterpillars are below half_cp
    df_half_bottom = df[df.Caterpillars <= half_cp]
    if df_half_bottom.shape[0] == 0 or df_half_bottom.Caterpillars.max() == 0.0:
        if verbose:
            print("WARNING: no bottom half found"
                f" for TubeID {df.TubeID.values[0]} with max_cp {max_cp}")
        return np.nan
    
    else:
        count_bottom = df_half_bottom.Caterpillars.max()
        day_bottom = df_half_bottom[df_half_bottom.Caterpillars == count_bottom].AprilDay.values[0]

    if verbose:
        print(f"TubeID {df.TubeID.values[0]}: max_cp {max_cp}, half_cp {half_cp}, "
              f"day_bottom {day_bottom} (count_bottom {count_bottom}), "
              f"day_top {day_top} (count_top {count_top})")
        
    if count_top == count_bottom:
        if verbose:
            print("WARNING: count_top == count_bottom"
                  f" for TubeID {df.TubeID.values[0]} with max_cp {max_cp}")
        return day_top
    
    # linear interpolation to find the day where the count is exactly half_cp
    d50 = day_bottom + (half_cp - count_bottom) * (day_top - day_bottom) / (count_top - count_bottom)
    return d50

In [50]:
df_raw[df_raw.TubeID == 1048]

,YearCatch,YearHatch,AreaShortName,Site,Tree,NovemberDate,TubeID,ExperimentName,AprilDay,Caterpillars
4658,1997,1998,DO,4,1,33,1048,CG1998,-11,0.0
4659,1997,1998,DO,4,1,33,1048,CG1998,-6,0.0
4660,1997,1998,DO,4,1,33,1048,CG1998,-1,0.0
4661,1997,1998,DO,4,1,33,1048,CG1998,6,2.0
4662,1997,1998,DO,4,1,33,1048,CG1998,9,2.0
4663,1997,1998,DO,4,1,33,1048,CG1998,14,5.0
4664,1997,1998,DO,4,1,33,1048,CG1998,17,9.0
4665,1997,1998,DO,4,1,33,1048,CG1998,20,0.0
4666,1997,1998,DO,4,1,33,1048,CG1998,24,0.0
4667,1997,1998,DO,4,1,33,1048,CG1998,27,0.0


In [51]:
calculate_d50(df_raw[df_raw.TubeID == 1048], verbose=1)

TubeID 1048: max_cp 9, half_cp 4.5, day_bottom 6 (count_bottom 2.0), day_top 14 (count_top 5.0)


np.float64(12.666666666666668)

In [ ]:
df_proc[np.logical_and(df_proc.YearHatch == 1998, df_proc.Tree == 1)]

,YearCatch,YearHatch,AreaShortName,Site,Tree,NovemberDate,TubeNumber,D50Calc
415,1997,1998,WA,2,1,20,6,"4,2"
419,1997,1998,OH,4,1,21,10,"-0,3"
420,1997,1998,OH,4,1,21,11,"2,4"
421,1997,1998,OH,4,1,21,12,"1,7"
428,1997,1998,DO,1,1,27,19,9
431,1997,1998,DO,4,1,27,24,"8,4"
443,1997,1998,OH,4,1,27,39,"2,9"
444,1997,1998,OH,4,1,27,40,"4,6"
453,1997,1998,WA,1,1,27,50,"3,9"
454,1997,1998,WA,1,1,27,52,"8,8"


In [52]:
df_proc[np.logical_and(df_proc.TubeNumber == 70, df_proc.YearHatch == 1998)]

,YearCatch,YearHatch,AreaShortName,Site,Tree,NovemberDate,TubeNumber,D50Calc
463,1997,1998,DO,4,1,33,70,14
